# Unit 9 exercises: feature towers and item cold-start

Run the recsys data generators before you begin. Unit 8's two-tower was the best path in the book so far (hit@10 ~0.34), but it gives every book a bare per-id `torch.nn.Embedding` row -- and that row is only informative for a book the training log has positive reads for. A **cold item** (zero train positives) never appears as a positive and, under Unit 8's uniform-over-the-full-catalog negative sampler, is only ever sampled as a *negative*, so its id row is pushed down and the book is buried. Measured on this data, the ID-only two-tower and item-item CF put **exactly zero** of the 818 zero-train books into any reader's top-10.

The **feature tower** keeps the two-tower's reader-tower / item-tower dot-product model but rebuilds the item tower from *what the book is*: a per-id embedding PLUS a learned genre embedding, a learned author embedding, and a `Linear` projection of the book's Unit-7 GloVe keyword vector, all summed into the shared space. Because the genre / author / GloVe parameters are shared across all books and trained from the warm items, a cold book inherits a sensible vector from its features even though its own id row only ever receives negative gradients. These exercises build the feature inputs, train and register `FeatureTowerRetrievalPath`, read its validation scoreboard, and measure the metric that matters here -- **cold-item coverage**.

Read it **honestly**. Measured at the pinned config (seed 0, k=10, 40 epochs, `negative_pool="warm"`), the feature tower scores warm hit@10 ~0.298 -- a little *below* the ID-only two-tower's ~0.340 (within 0.06), and NOT a new warm record. The win is elsewhere: it lifts cold-item coverage from 0 to ~0.138 (113 of the 818 zero-train books surfaced), while CF and the ID-only two-tower stay at exactly 0. It is a warm/cold **compromise**, not dominance -- and it is not even the *first* path to serve cold items (Unit 7's semantic content path already covers ~0.328 of them). The honest headline is narrower and true: the feature tower is the first **learned-taste / collaborative** path to reach cold items while holding near the book's best warm hit.

Everything is measured on the **validation** split (`val`); each reader's **seen** set is their train-positive history. The cold-coverage denominator and reader cohort are built from TRAIN positives only -- no held-out dependency. The `test` holdout stays sealed (the designated cold items have their relevance deferred to Checkpoint B): no cell here may touch it. Each answer is left for you to complete; no solutions live here.

A speed note, because training recurs. A full feature-tower fit at the pinned 40 epochs takes ~20-27 s; the exercises fit it a few times and **reuse** the fitted objects, and every training cell stays well under the 120 s/cell budget. The two Challenge exercises each do one extra full fit; the determinism cell uses a deliberately small epoch count (reproducibility holds at any epoch count).

In [ ]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    FeatureTowerRetrievalPath,
    ItemItemRetrievalPath,
    PathRegistry,
    SemanticEmbeddingRetrievalPath,
    TwoTowerRetrievalPath,
    book_embedding,
    run_validation_scoreboard,
)
from bookrec.lexical import tokenize   # the library tokenizer (what book_embedding uses internally)

SEED = 0
K = 10        # every scoreboard reports at the top-10 cutoff

# --- the real interaction log + content (the same substrate as Units 1-8) ------------------
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")        # {item_id: Book} (2000 books)
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")     # {item_id: "space joined token bag"}
glove = bookrec.load_glove_subset()                                # the committed GloVe subset

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])      # 60 readers the personalized paths cannot serve

# The interaction log as plain row dicts -- the row-mapping log the paths duck-type.
interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")

# Each reader's train-positive history is their `seen` (exclusion) set; their val positives are the
# relevance target. We grade on `val`; no cell here may read the sealed `test` split.
seen_by = defaultdict(set)
val_pos_by = defaultdict(set)
train_pos_items = set()
for row in rows:
    if row["label"] != 1:
        continue
    if row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])
        train_pos_items.add(row["item_id"])
    elif row["split"] == "val":
        val_pos_by[row["reader_id"]].add(row["item_id"])

# The 818 zero-train books -- books with NO train positive. This is the cold-coverage denominator,
# and it is built from TRAIN positives alone (no val / test dependency).
zero_train = set(catalog_ids) - train_pos_items

## Exercise 1 - A cold book has no history, but it still has features

The whole idea of the feature tower is that a book is more than its id. Unit 8 gave every book a bare per-id embedding, which carries no useful signal for a book nobody has read -- its id row is only ever a sampled negative. The feature tower instead feeds the item tower *content*: the book's **genre(s)** (a multi-hot over the genre vocabulary), its **author**, and its **GloVe keyword vector** (the Unit-7 mean-pooled embedding). None of these needs a single interaction -- they come straight from the catalog -- so a cold book gets exactly the same kind of feature description as a popular one.

Build those feature inputs by hand for two books: a warm book (one with train interactions) and a **cold** book (one from `zero_train`). Build the catalog-wide genre vocabulary (sorted, exactly as the tower does), a `genre_multihot` for each book over that vocabulary, read each book's author id, and compute each book's GloVe keyword vector with the shipped `book_embedding`. Then confirm the punchline: the cold book, despite zero train positives, still carries genres and a nonzero GloVe vector. In two or three sentences, explain why these shared, content-derived features can place a cold book sensibly where a per-id embedding (shaped for the cold book only by negative sampling) cannot.

In [ ]:
# The catalog-wide genre vocabulary, sorted for determinism (exactly what the feature tower builds).
genre_vocab = sorted({
    token.strip()
    for item_id in catalog_ids
    for token in catalog[item_id].fields.get("genres", "").split(";")
    if token.strip()
})
genre_index = {genre: j for j, genre in enumerate(genre_vocab)}


def book_genres(item_id):
    """The book's genre tokens, parsed from the catalog's ';'-joined `genres` field."""
    raw = catalog[item_id].fields.get("genres", "")
    return [token.strip() for token in raw.split(";") if token.strip()]


def genre_multihot(item_id):
    """A multi-hot vector over genre_vocab: 1.0 in each slot the book's genres occupy."""
    vec = np.zeros(len(genre_vocab), dtype=np.float32)
    # TODO: for genre in book_genres(item_id): vec[genre_index[genre]] = 1.0
    return vec


warm_book = min(train_pos_items)   # a book WITH train interactions (smallest id with a train positive)
cold_book = min(zero_train)     # a zero-train (cold) book -- NO train positives (only ever a sampled negative)

warm_genres = None    # TODO: book_genres(warm_book)                    -- a list of genre tokens
cold_genres = None    # TODO: book_genres(cold_book)                    -- the cold book still HAS genres
warm_author = None    # TODO: catalog[warm_book].fields.get("author_id", "")
cold_author = None    # TODO: catalog[cold_book].fields.get("author_id", "")
warm_glove = None     # TODO: book_embedding(tokenize(keywords[warm_book]), glove)  -- GloVe keyword vector
cold_glove = None     # TODO: book_embedding(tokenize(keywords[cold_book]), glove)

warm_multihot = None  # TODO: genre_multihot(warm_book)   -- after you fill in genre_multihot above
cold_multihot = None  # TODO: genre_multihot(cold_book)

# The punchline: the cold book has zero train positives, yet it still carries content features.
cold_in_zero_train = None    # TODO: bool - cold_book in zero_train
cold_has_features = None     # TODO: bool - len(cold_genres) > 0 and bool(np.any(cold_glove != 0.0))

feature_explanation = "TODO"  # 2-3 sentences: genre / author / GloVe are shared, content-derived
                              # features trained across ALL warm books, so a cold book inherits a
                              # sensible vector from them -- a bare per-id embedding has nothing to
                              # learn a taste direction from for a book with no train positives

## Exercise 2 - Train and register the feature tower (warm-only negatives)

Now the real path. `FeatureTowerRetrievalPath` trains a reader id-embedding tower and the composed item tower (id + genre + author + GloVe projection) by BPR + Adam + weight decay, exactly like Unit 8's two-tower but with the richer item side. Features arrive through the **constructor** (the catalog `Book` records, the keyword corpus and the GloVe subset), so `fit(rows, catalog=catalog_ids)` keeps the exact protocol signature -- `catalog` is an iterable of item ids, never `Book`s. The pinned defaults are the configuration measured on this data: `embedding_dim=32, n_epochs=40, learning_rate=0.01, n_negatives=10, batch_size=256, weight_decay=1e-4, negative_pool="warm", seed=0`.

The one design choice that matters most is `negative_pool`. The shipped default `"warm"` draws negatives **only from the train-positive (warm) item universe**, so a cold book is never sampled as a negative and its features are free to place it. (Exercise 7 shows what the `"full"` sampler does instead.) Fit the path once at the default config (~20-27 s) and keep the fitted object -- Exercises 3, 4, 5 and 6 reuse it, do not refit there. Read `loss_history` and confirm the BPR loss fell from the first epoch to the last, register the fitted path in a `PathRegistry` (it must own the name `feature-tower-v1`), and confirm it serves a warm reader `K` recommendations. In two or three sentences, explain what the item tower now combines that Unit 8's did not, and why that is what lets a cold book score at all.

In [ ]:
# One full fit at the pinned warm-only config (~20-27 s). REUSED by Exercises 3, 4, 5, 6 -- do not refit.
feature_tower = FeatureTowerRetrievalPath(
    catalog, keywords, glove, negative_pool="warm", seed=SEED
).fit(rows, catalog=catalog_ids)

losses = None             # TODO: feature_tower.loss_history   -- mean BPR loss per epoch (a list)
n_epochs_recorded = None  # TODO: len(losses)                  -- 40, the pinned default
loss_decreased = None     # TODO: bool - float(losses[-1]) < float(losses[0])

# Register it as a first-class recommender (it must own the name 'feature-tower-v1').
registry = PathRegistry()
registry.register(feature_tower)
artifact_name = None      # TODO: feature_tower.artifact_name()   -- 'feature-tower-v1'
registered = None         # TODO: bool - artifact_name == "feature-tower-v1" and "feature-tower" in registry

# A reader WITH a learned embedding gets K recs; `seen` only EXCLUDES already-read books.
warm_reader = None        # TODO: feature_tower.artifact()["reader_ids"][0]
warm_recs = None          # TODO: [c.item_id for c in feature_tower.retrieve(warm_reader, {"seen": seen_by[warm_reader]}, K)]
got_k_recs = None         # TODO: bool - len(warm_recs) == K

tower_explanation = "TODO"  # 2-3 sentences: the item tower now SUMS a per-id embedding with learned
                            # genre + author embeddings and a Linear projection of the GloVe keyword
                            # vector; the shared feature parameters give a cold book a vector even
                            # though its id row only ever saw negative gradients

## Exercise 3 - The validation scoreboard: a small, honest warm cost

Score the feature tower beside the paths it extends. Fit Unit 8's ID-only two-tower (its shipped default config), Unit 4's item-item CF, and Unit 7's semantic content path, and measure hit@10 on `val` for all four (reuse the `feature_tower` from Exercise 2 -- do not refit it). Keep these three fitted objects: Exercise 4 reuses them for the coverage comparison.

Read the numbers **honestly, and the honest reading is NOT a warm win**. The feature tower should land around hit@10 ~0.298 -- a little **below** the ID-only two-tower's ~0.340 (within 0.06), above CF (~0.252) and far above semantic (~0.102). So the feature tower does NOT set a warm record; it pays a few points of warm hit relative to the plain two-tower. That trade buys something the next exercise measures (cold-item reach). In three or four sentences, state the warm ordering plainly, say clearly that the feature tower is within 0.06 of the best but does not beat it, and preview *why* a path might accept a small warm cost on purpose.

In [ ]:
# Fit the comparison paths. The two-tower is the only other slow fit (~13 s); CF and semantic are fast.
# The feature_tower itself is REUSED from Exercise 2 (no refit). KEEP these three for Exercise 4.
two_tower = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)       # Unit 8 ID-only
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)                # Unit 4 CF
semantic = SemanticEmbeddingRetrievalPath(keywords, glove).fit(rows, catalog=catalog_ids)  # Unit 7


def val_hit(path):
    return run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    ).hit_rate_at_k


ft_hit = None      # TODO: val_hit(feature_tower)   -- ~0.298 (a small cost vs the two-tower)
tt_hit = None      # TODO: val_hit(two_tower)       -- ~0.340 (the warm leader)
cf_hit = None      # TODO: val_hit(item_item)       -- ~0.252
sem_hit = None     # TODO: val_hit(semantic)        -- ~0.102

feature_tower_not_best = None   # TODO: bool - ft_hit < tt_hit    (NOT a warm record)
within_six_points = None        # TODO: bool - ft_hit >= tt_hit - 0.06   (but close -- not tanked)
beats_cf = None                 # TODO: bool - ft_hit > cf_hit

honest_warm_reading = "TODO"  # 3-4 sentences: the warm ordering (two-tower > feature tower > CF >
                              # semantic); the feature tower is within 0.06 of the best but does NOT
                              # beat it; it trades a little warm hit on purpose -- for what Exercise 4
                              # measures

## Exercise 4 - Cold-item coverage: the metric that matters here

The designated cold items have no `val` positives by construction, so cold **relevance** cannot be graded on `val` (only the sealed `test` could, at Checkpoint B). The validation-safe metric is cold-item **coverage**: of the 818 zero-train books, how many appear in *some* fitted reader's top-10. Crucially the cohort and denominator are both built from TRAIN alone -- score every fitted non-cold reader from their train history and count the unique zero-train ids that land in a top-10, over all 818. This never depends on who happens to have a val positive.

Write a `cold_coverage(path)` helper and measure it for the feature tower, the ID-only two-tower, item-item CF, and Unit 7's semantic path (reuse all four fitted objects). You should find the ID-only two-tower and CF at **exactly 0** (their cold items get negative-shaped / unseen embeddings and are buried), the feature tower at ~0.138 (113/818), and semantic **high** at ~0.328. In three or four sentences, give the coverage ordering and the honest headline: the feature tower is NOT the first path to surface cold items -- Unit 7's content path already does, freely -- it is the first *learned-taste / collaborative* path to reach cold items while holding near the book's best warm hit, a compromise rather than dominance.

In [ ]:
# Cohort = ALL fitted non-cold readers, scored from TRAIN history only (no val dependency).
def fitted_cohort(path):
    try:                                     # the tower paths expose fitted reader ids in the artifact
        fitted = [int(r) for r in path.artifact()["reader_ids"]]
    except KeyError:                         # item-item CF has no reader embedding -> use the train cohort
        fitted = sorted(seen_by)
    return [r for r in fitted if r not in cold_readers]


def cold_coverage(path):
    """(coverage fraction, count): unique zero-train ids in any top-10 / 818, train-only scoring."""
    surfaced = set()
    for reader_id in fitted_cohort(path):
        pass
        # TODO: for c in path.retrieve(reader_id, {"seen": seen_by[reader_id]}, K):
        # TODO:     if c.item_id in zero_train: surfaced.add(c.item_id)
    return len(surfaced) / len(zero_train), len(surfaced)


denom = None          # TODO: len(zero_train)                 -- 818, the validation-free denominator
ft_cov, ft_n = None, None     # TODO: cold_coverage(feature_tower)    -- ~0.138 (113)
tt_cov, tt_n = None, None     # TODO: cold_coverage(two_tower)        -- exactly 0.0 (0)
cf_cov, cf_n = None, None     # TODO: cold_coverage(item_item)        -- exactly 0.0 (0)
sem_cov, sem_n = None, None   # TODO: cold_coverage(semantic)         -- ~0.328 (content already serves cold)

id_only_and_cf_bury_cold = None  # TODO: bool - tt_cov == 0.0 and cf_cov == 0.0
feature_tower_reaches_cold = None  # TODO: bool - ft_cov > 0.0 and ft_cov > tt_cov
semantic_already_serves_cold = None  # TODO: bool - sem_cov > ft_cov

coverage_reading = "TODO"  # 3-4 sentences: coverage ordering (semantic > feature tower > two-tower =
                           # CF = 0); the honest headline -- NOT the first to serve cold (Unit 7
                           # content already does), but the first LEARNED/collaborative path to reach
                           # cold while holding near the best warm hit -- a compromise, not dominance

## Exercise 5 - One concrete cold book the feature tower surfaces (that the two-tower buries)

Coverage is a count; make it concrete for one reader. Among the readers who are not designated cold readers, some have a `val` positive that is a zero-train book they went on to read next -- there are exactly 97 such **incidental-cold-val** readers on this data. Build that cohort deterministically (sorted), then find a reader in it for whom the feature tower surfaces, inside its top-10, one of that reader's own cold `val` positives -- a cold book they actually read next. (This is the report-only cold hit; it is small, measured ~0.010, so possibly just one such reader exists (one, on this data) -- pick the smallest reader id among them.)

For that reader, show the surfaced cold book's id and title, and confirm the honest contrast: the **same** reader's ID-only two-tower top-10 and item-item CF top-10 contain **no** zero-train book at all. In two or three sentences, explain why the feature tower can rank a never-read book into a reader's top-10 while the two-tower and CF, whose id row for that book was shaped only by negative sampling (CF has zero co-occurrence), cannot.

In [ ]:
# The 97 incidental-cold-val readers: non-cold readers with an UNSEEN zero-train val positive.
incidental_cohort = sorted(
    reader_id
    for reader_id in val_pos_by
    if reader_id not in cold_readers
    and ((val_pos_by[reader_id] - seen_by[reader_id]) & zero_train)
)
cohort_size = None   # TODO: len(incidental_cohort)   -- 97


def surfaced_cold_val_positive(path, reader_id):
    """A cold val-positive book this path puts in the reader's top-10, or None."""
    cold_relevant = (val_pos_by[reader_id] - seen_by[reader_id]) & zero_train
    top = {c.item_id for c in path.retrieve(reader_id, {"seen": seen_by[reader_id]}, K)}
    hit = top & cold_relevant
    return min(hit) if hit else None


# The deterministic reader: the smallest-id cohort reader whose feature tower surfaces a cold book
# they actually read next.
example_reader = None   # TODO: next(r for r in incidental_cohort if surfaced_cold_val_positive(feature_tower, r) is not None)
surfaced_book = None    # TODO: surfaced_cold_val_positive(feature_tower, example_reader)
surfaced_title = None   # TODO: catalog[surfaced_book].title
surfaced_is_cold = None # TODO: bool - surfaced_book in zero_train

# The honest contrast on the SAME reader: the two-tower and CF surface NO zero-train book at all.
def cold_in_top(path, reader_id):
    return [c.item_id for c in path.retrieve(reader_id, {"seen": seen_by[reader_id]}, K) if c.item_id in zero_train]

tt_cold_for_reader = None   # TODO: cold_in_top(two_tower, example_reader)    -- expect []
cf_cold_for_reader = None   # TODO: cold_in_top(item_item, example_reader)    -- expect []
only_feature_tower_surfaces_cold = None  # TODO: bool - len(tt_cold_for_reader) == 0 and len(cf_cold_for_reader) == 0

cold_example_explanation = "TODO"  # 2-3 sentences: the feature tower scores the cold book from its
                                   # genre/author/GloVe features (shared, trained on warm books), so
                                   # it can rank a never-read book high; the two-tower / CF know the
                                   # cold book only as a sampled negative / zero co-occurrence, so they bury it

## Exercise 6 - Determinism and the retrieve contract

A trained model you cannot reproduce is not science, and a retrieval path has a contract to honor. `FeatureTowerRetrievalPath.fit` seeds `torch.manual_seed` and a numpy RNG, enables `torch.use_deterministic_algorithms(True)` and pins `torch.set_num_threads(1)` (saving and restoring the last two process-global switches), so two fits at the same seed agree. Determinism holds at any epoch count, so fit **two** small towers here (a handful of epochs, a few seconds each) rather than two full 40-epoch fits -- the reproducibility lesson is the same and the cell stays cheap.

Confirm the two small fits are `allclose` on both learned matrices and give an **identical** top-K ranking for a few readers (that tolerance-plus-ranking match is the gate; bit-identical floats are a nice CPU bonus, not the standard). Then check the retrieve contract on the `feature_tower` from Exercise 2: a known reader with an empty `seen` still gets `K` recs, and an unknown reader gets `[]`. In two or three sentences, explain why we gate on identical ranking plus `allclose` rather than demanding exactly equal floats.

In [ ]:
# Determinism holds at any epoch count, so use a SMALL n_epochs here (two cheap fits, a few seconds
# each) instead of two full 40-epoch fits -- the reproducibility gate is identical.
SMALL_EPOCHS = 5
fit_a = FeatureTowerRetrievalPath(catalog, keywords, glove, n_epochs=SMALL_EPOCHS, seed=SEED).fit(rows, catalog=catalog_ids)
fit_b = FeatureTowerRetrievalPath(catalog, keywords, glove, n_epochs=SMALL_EPOCHS, seed=SEED).fit(rows, catalog=catalog_ids)

readers_allclose = None  # TODO: bool - np.allclose(fit_a.reader_embeddings, fit_b.reader_embeddings)
items_allclose = None    # TODO: bool - np.allclose(fit_a.item_embeddings, fit_b.item_embeddings)

# Identical top-K ranking for the first few readers (the reproducibility gate).
reader_ids = None            # TODO: fit_a.artifact()["reader_ids"][:5]
rankings_identical = None    # TODO: bool - all(
#     [c.item_id for c in fit_a.retrieve(r, {"seen": set()}, K)]
#     == [c.item_id for c in fit_b.retrieve(r, {"seen": set()}, K)]
#     for r in reader_ids
# )
bit_identical = None   # TODO: bool - np.array_equal(fit_a.item_embeddings, fit_b.item_embeddings)  (bonus, not the gate)

# The retrieve contract, on the fitted `feature_tower` from Exercise 2.
known_reader = None       # TODO: feature_tower.artifact()["reader_ids"][0]
empty_seen_recs = None    # TODO: feature_tower.retrieve(known_reader, {"seen": set()}, K)
known_gets_k = None       # TODO: bool - len(empty_seen_recs) == K
unknown_reader = None     # TODO: next(r for r in range(10_000_000) if r not in set(feature_tower.artifact()["reader_ids"]))
unknown_gets_empty = None # TODO: bool - feature_tower.retrieve(unknown_reader, {"seen": set()}, K) == []

determinism_explanation = "TODO"  # 2-3 sentences: we gate on identical top-K RANKING + allclose
                                  # matrices (tolerance-based), not exact floats, because tiny float
                                  # differences that never change a ranking are harmless

## Exercise 7 - Challenge: the negative pool is the whole mechanism

Here is the decisive design choice. 41% of this catalog is zero-train, so if negatives are sampled uniformly over the **full** catalog (Unit 8's sampler), a cold book is *only ever* a negative: the tower gets a push-down gradient on it every epoch and no positive to balance it, and features can only weakly counter a book's own negative gradient (full-pool coverage collapses to ~0.048, not to 0). Drawing negatives only from the **warm** (train-positive) universe removes that push-down, so a cold book's features are free to place it. The knob is `negative_pool`, and it is an honest **warm-cold trade**, not a free lunch.

Refit the feature tower once with `negative_pool="full"` (everything else pinned; ~20 s) and compare it to the warm-only `feature_tower` from Exercise 2 on BOTH axes -- warm hit@10 and cold coverage. You should find that `"full"` *raises* warm hit a touch (~0.338, almost the plain two-tower) while *collapsing* cold coverage (~0.048, 39/818), and `"warm"` gives up a little warm (~0.298) to roughly triple cold coverage (~0.138). In three or four sentences, describe the trade in both directions and explain the mechanism -- why making cold items negatives-only suppresses their reach so hard (coverage falls to ~0.048, not to 0) -- their features can only weakly counter a pure push-down gradient.

In [ ]:
# One extra full fit (~20 s): the SAME tower with Unit 8's full-catalog negative sampler.
# The warm-only `feature_tower` from Exercise 2 is REUSED -- do not refit it.
full_pool_tower = FeatureTowerRetrievalPath(
    catalog, keywords, glove, negative_pool="full", seed=SEED
).fit(rows, catalog=catalog_ids)

# Reuse val_hit (Exercise 3) and cold_coverage (Exercise 4).
warm_pool_hit = None    # TODO: val_hit(feature_tower)        -- ~0.298
full_pool_hit = None    # TODO: val_hit(full_pool_tower)      -- ~0.338 (higher warm)
warm_pool_cov, _ = None, None   # TODO: cold_coverage(feature_tower)      -- ~0.138 (113)
full_pool_cov, _ = None, None   # TODO: cold_coverage(full_pool_tower)    -- ~0.048 (39)  cold collapses

full_raises_warm = None      # TODO: bool - full_pool_hit > warm_pool_hit
warm_raises_cold = None      # TODO: bool - warm_pool_cov > full_pool_cov

negative_pool_explanation = "TODO"  # 3-4 sentences: full-catalog negatives make the 818 cold items
                                    # negatives-only -> a constant push-down with no balancing
                                    # positive, so their features can only weakly counter it (cold cov collapses to ~0.048, not 0)
                                    # while warm edges up; warm-only negatives reverse the trade

## Exercise 8 - Challenge: hard negatives trade even harder (and the false-negative caveat)

A tempting idea is to make the negatives *harder* -- instead of uniform-over-warm, sample popular warm books (which a reader is more likely to have plausibly considered) more often. `negative_pool="hard"` does exactly that (popularity-weighted over the warm universe). The folklore is that harder negatives *sharpen* a ranker; the honest measurement on this data is that they do **not** sharpen both axes -- they push the warm-cold trade even further toward cold, at a real warm cost. There is also a correctness caveat: a popular book a reader has not logged might actually be a true (unobserved) positive, so hard sampling risks training against books the reader would in fact like (a **false negative**).

Refit once with `negative_pool="hard"` (everything else pinned; ~24 s) and compare it to the uniform warm-only `feature_tower` from Exercise 2 on warm hit@10 and cold coverage. You should find hard negatives *lower* warm hit (~0.172, well below the uniform ~0.298) while *raising* cold coverage further (~0.226, 185/818). In three or four sentences, state the measured trade, say plainly that hard negatives did not sharpen both axes here, and explain the false-negative risk that makes aggressive hard sampling something to use with care.

In [ ]:
# One extra full fit (~24 s): popularity-weighted ('hard') negatives over the same warm universe.
# The uniform warm-only `feature_tower` from Exercise 2 is REUSED -- do not refit it.
hard_pool_tower = FeatureTowerRetrievalPath(
    catalog, keywords, glove, negative_pool="hard", seed=SEED
).fit(rows, catalog=catalog_ids)

uniform_hit = None    # TODO: val_hit(feature_tower)       -- ~0.298 (uniform warm-only)
hard_hit = None       # TODO: val_hit(hard_pool_tower)     -- ~0.172 (lower warm)
uniform_cov, _ = None, None   # TODO: cold_coverage(feature_tower)      -- ~0.138 (113)
hard_cov, _ = None, None      # TODO: cold_coverage(hard_pool_tower)    -- ~0.226 (185)  cold rises further

hard_lowers_warm = None     # TODO: bool - hard_hit < uniform_hit
hard_raises_cold = None     # TODO: bool - hard_cov > uniform_cov
hard_did_not_sharpen = None # TODO: bool - hard_lowers_warm and hard_raises_cold   (traded, not sharpened)

hard_negative_explanation = "TODO"  # 3-4 sentences: hard negatives traded warm (~0.298 -> ~0.172) for
                                    # more cold reach (~0.138 -> ~0.226) -- they did NOT sharpen both;
                                    # and a popular un-logged book may be a true positive, so hard
                                    # sampling risks training against false negatives